# KV Cache — Interactive Demo

This notebook walks you through the key ideas behind **KV Cache** in Transformer-based language models:

1. Why caching keys and values speeds up autoregressive generation
2. How much memory KV Cache consumes
3. Variants: **Multi-query Attention**, **Group-query Attention**, **Multi-head Latent Attention**
4. **Sliding Window Attention** and **Streaming LLM**
5. **Pruning** the KV Cache (Scissorhands / H2O)
6. **Cross-conversation** cache reuse

Run every cell from top to bottom; interactive widgets appear where you see sliders or buttons.

## Learning Objectives

After finishing this notebook you should be able to:

- Explain why KV Cache reduces redundant computation during decoding.
- Estimate the memory footprint of a KV Cache for a given model and sequence length.
- Compare MHA, MQA, GQA, and MLA and articulate their trade-offs.
- Describe how sliding-window and pruning-based methods shrink the cache at runtime.

## 0 · Setup

In [ ]:
import math, textwrap
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('Setup complete ✓')

---
## 1 · Scaled Dot-Product Attention (recap)

Given token embeddings $x_1, \ldots, x_T$ we compute:

$$
Q = XW_Q, \quad K = XW_K, \quad V = XW_V
$$
$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

During **autoregressive generation** the model produces one token at a time.  
At step $t$ the new query is $q_t$ but the keys/values for tokens $1 \ldots t-1$ are **identical** to what was computed in previous steps — without KV Cache we recompute them from scratch every step.

In [ ]:
def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.T / math.sqrt(d_k)      # (T_q, T_k)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = softmax(scores)
    return weights @ V, weights

# Tiny demo: 4 tokens, d_model=8
rng = np.random.default_rng(42)
T, d = 4, 8
X = rng.standard_normal((T, d))
Wq = rng.standard_normal((d, d)) * 0.3
Wk = rng.standard_normal((d, d)) * 0.3
Wv = rng.standard_normal((d, d)) * 0.3

Q = X @ Wq
K = X @ Wk
V = X @ Wv

causal_mask = np.tril(np.ones((T, T), dtype=bool))
output, weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

fig, ax = plt.subplots(figsize=(4, 3.5))
im = ax.imshow(weights, cmap='Blues', vmin=0, vmax=1)
ax.set_title('Causal attention weights\n(4-token sequence)')
ax.set_xlabel('Key position'); ax.set_ylabel('Query position')
ax.set_xticks(range(T)); ax.set_yticks(range(T))
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

---
## 2 · KV Cache — Eliminating Redundant Computation

**Without cache:** generating token $t$ requires computing $K$ and $V$ for all $t$ positions → $O(t)$ matmuls per step → $O(T^2)$ total work.

**With cache:** $K$ and $V$ up to position $t-1$ are stored from the previous step.  
Only the *new* key/value pair is computed and appended → one matmul per step → $O(T)$ total work.

The animation below counts how many key/value *rows* need to be computed at each decoding step.

In [ ]:
def draw_kvcache_comparison(step, total_steps=8):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    colors_cached = ['#4CAF50'] * step + ['#E0E0E0'] * (total_steps - step)
    colors_nocache = ['#F44336'] * step + ['#E0E0E0'] * (total_steps - step)

    for ax, title, cols, cost_fn, cost_label in [
        (axes[0], 'WITHOUT KV Cache', colors_nocache,
         lambda s: s, f'KV rows computed at step {step}: {step}'),
        (axes[1], 'WITH KV Cache', colors_cached,
         lambda s: 1,  f'KV rows computed at step {step}: 1 (new token only)'),
    ]:
        for i in range(total_steps):
            ax.barh(i, 1, color=cols[i], edgecolor='white', linewidth=1.5)
        ax.set_title(title, fontweight='bold')
        ax.set_xlim(0, 1.1)
        ax.set_yticks(range(total_steps))
        ax.set_yticklabels([f'tok {i+1}' for i in range(total_steps)])
        ax.set_xlabel('In memory / computed')
        ax.text(0.5, -1.5, cost_label, ha='center', color='#333', fontsize=10,
                transform=ax.get_yaxis_transform())

    total_no_cache = step * (step + 1) // 2
    total_cache    = step
    fig.suptitle(
        f'Decoding step {step}/{total_steps}  |  '
        f'Total KV computations so far — no cache: {total_no_cache}   cache: {total_cache}',
        fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

step_slider = widgets.IntSlider(value=1, min=1, max=8, step=1,
                                description='Decode step:', continuous_update=False,
                                style={'description_width': 'initial'})
widgets.interact(draw_kvcache_comparison, step=step_slider, total_steps=widgets.fixed(8))

---
## 3 · KV Cache Memory Footprint

For a transformer with:
- $L$ layers
- $H$ attention heads
- $d_{head}$ head dimension
- $P$ bytes per parameter (e.g. 2 for FP16)
- $T$ tokens in the KV cache

Total KV cache size:
$$
\text{Size} = 2 \times L \times H \times d_{head} \times P \times T \text{ bytes}
$$
(factor 2 = one K tensor + one V tensor)

**Example — Gemma 2:** $L{=}46$, $H{=}32$, $d_{head}{=}128$, FP16 → **736 KB per token**.

In [ ]:
out_memory = widgets.Output()

w_layers  = widgets.IntSlider(value=46, min=1,  max=128, description='Layers (L):')
w_heads   = widgets.IntSlider(value=32, min=1,  max=128, description='Heads (H):')
w_dhead   = widgets.IntSlider(value=128,min=16, max=512, step=16, description='d_head:')
w_prec    = widgets.Dropdown(options=[('FP32 (4 B)', 4), ('FP16 (2 B)', 2), ('INT8 (1 B)', 1)],
                             value=2, description='Precision:')
w_tokens  = widgets.IntSlider(value=4096, min=128, max=128000, step=128,
                              description='Seq len (T):')
w_gpu_gb  = widgets.FloatSlider(value=80, min=8, max=640, step=8, description='GPU VRAM (GB):')

def update_memory(layers, heads, dhead, precision_bytes, tokens, gpu_gb):
    per_token_bytes = 2 * layers * heads * dhead * precision_bytes
    total_bytes     = per_token_bytes * tokens
    gpu_bytes       = gpu_gb * 1024**3
    max_tokens      = int(gpu_bytes / per_token_bytes)

    with out_memory:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

        # Bar: bytes breakdown
        labels = ['Per-token\n(KB)', 'Total cache\n(MB)', 'GPU capacity\n(GB×1000 MB)']
        vals   = [per_token_bytes/1024, total_bytes/1024**2, gpu_gb*1000]
        colors = ['#1976D2', '#388E3C', '#FFA000']
        axes[0].bar(labels, vals, color=colors, edgecolor='white', width=0.5)
        axes[0].set_title('Memory sizes (different units!)')
        for i, v in enumerate(vals):
            axes[0].text(i, v * 1.02, f'{v:,.1f}', ha='center', fontsize=9)

        # Donut: fraction of GPU used
        frac = min(total_bytes / gpu_bytes, 1.0)
        axes[1].pie([frac, 1-frac], labels=['KV Cache', 'Other GPU'],
                    colors=['#1976D2', '#E0E0E0'], startangle=90,
                    autopct='%1.1f%%', wedgeprops={'edgecolor':'white','linewidth':1.5})
        axes[1].set_title(f'KV cache as fraction of {gpu_gb:.0f} GB GPU')

        fig.suptitle(
            f'Per-token: {per_token_bytes/1024:.1f} KB   |   '
            f'Total ({tokens:,} tokens): {total_bytes/1024**2:.1f} MB   |   '
            f'Max tokens on GPU: {max_tokens:,}',
            fontsize=11, fontweight='bold')
        plt.tight_layout()
        plt.show()

ui = widgets.VBox([
    widgets.HBox([w_layers, w_heads]),
    widgets.HBox([w_dhead, w_prec]),
    widgets.HBox([w_tokens, w_gpu_gb]),
    out_memory
])

widgets.interactive_output(update_memory, {
    'layers': w_layers, 'heads': w_heads, 'dhead': w_dhead,
    'precision_bytes': w_prec, 'tokens': w_tokens, 'gpu_gb': w_gpu_gb
})
display(ui)

---
## 4 · Attention Variants: MHA → MQA → GQA → MLA

| Method | Keys/Values | KV Cache size | Model quality |
|---|---|---|---|
| **MHA** – Multi-Head Attention | one K,V per head | full | best |
| **MQA** – Multi-Query Attention | shared single K,V | ÷ heads | may degrade |
| **GQA** – Group-Query Attention | one K,V per group | ÷ group_size | good trade-off |
| **MLA** – Multi-head Latent Attention | compressed latent | very small | DeepSeek approach |

Use the widget below to visualise how the KV cache shrinks across variants.

In [ ]:
def draw_variants(n_heads=8, group_size=2, mla_rank=4, d_head=64, seq_len=512, prec_bytes=2):
    n_groups  = max(1, n_heads // group_size)
    mha_kv    = 2 * n_heads  * d_head * seq_len * prec_bytes / 1024      # KB
    mqa_kv    = 2 * 1        * d_head * seq_len * prec_bytes / 1024
    gqa_kv    = 2 * n_groups * d_head * seq_len * prec_bytes / 1024
    mla_kv    = 2 * mla_rank * seq_len * prec_bytes / 1024

    labels = ['MHA', 'GQA', 'MQA', 'MLA']
    sizes  = [mha_kv, gqa_kv, mqa_kv, mla_kv]
    colors = ['#1565C0', '#2E7D32', '#F57F17', '#6A1B9A']

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Bar chart
    bars = axes[0].bar(labels, sizes, color=colors, edgecolor='white', width=0.55)
    axes[0].set_ylabel('KV Cache (KB)')
    axes[0].set_title(f'KV Cache size — {n_heads} heads, seq={seq_len}, d_head={d_head}')
    for b, s in zip(bars, sizes):
        axes[0].text(b.get_x() + b.get_width()/2, b.get_height() * 1.02,
                     f'{s:.1f} KB', ha='center', fontsize=9)

    # Head diagram
    ax = axes[1]
    ax.set_xlim(-0.5, 4); ax.set_ylim(-0.5, n_heads + 0.5)
    ax.set_xticks([0, 1, 2, 3]); ax.set_xticklabels(['Q heads', 'K groups', 'V groups', ''])
    ax.set_title('Query heads → KV groups mapping')
    ax.axvline(0.5, color='gray', lw=0.5, ls='--')
    ax.axvline(1.5, color='gray', lw=0.5, ls='--')

    palette = plt.cm.tab10(np.linspace(0, 0.9, n_groups))
    for h in range(n_heads):
        g = h // group_size
        col = palette[g % len(palette)]
        ax.add_patch(mpatches.FancyBboxPatch((-.4, h-.4), 0.8, 0.8, color=col, alpha=0.85,
                                              boxstyle='round,pad=0.05'))
        ax.text(0, h, f'Q{h}', ha='center', va='center', fontsize=8, color='white', fontweight='bold')

    for g in range(n_groups):
        y_center = (g * group_size + (g+1) * group_size - 1) / 2
        col = palette[g % len(palette)]
        for x_pos, label in [(1, f'K{g}'), (2, f'V{g}')]:
            ax.add_patch(mpatches.FancyBboxPatch((x_pos-.4, y_center-.4), 0.8, 0.8,
                                                  color=col, alpha=0.7,
                                                  boxstyle='round,pad=0.05'))
            ax.text(x_pos, y_center, label, ha='center', va='center',
                    fontsize=8, color='white', fontweight='bold')
        # draw lines from each Q to its KV group
        for h in range(g * group_size, min((g+1) * group_size, n_heads)):
            ax.plot([0.4, 0.6], [h, y_center], color=col, lw=1, alpha=0.5)
    ax.axis('off')

    plt.tight_layout()
    plt.show()

    print(f'  Reduction vs MHA → GQA: ×{mha_kv/gqa_kv:.1f}  '
          f'MQA: ×{mha_kv/mqa_kv:.1f}  '
          f'MLA: ×{mha_kv/max(mla_kv,0.001):.1f}')

widgets.interact(
    draw_variants,
    n_heads   = widgets.IntSlider(value=8,   min=2,  max=32, step=2,  description='# Heads:'),
    group_size= widgets.IntSlider(value=2,   min=1,  max=8,  step=1,  description='Group size (GQA):'),
    mla_rank  = widgets.IntSlider(value=4,   min=1,  max=32, step=1,  description='MLA rank:'),
    d_head    = widgets.IntSlider(value=64,  min=16, max=256,step=16, description='d_head:'),
    seq_len   = widgets.IntSlider(value=512, min=64, max=8192,step=64,description='Seq len:'),
    prec_bytes= widgets.Dropdown(options=[('FP32',4),('FP16',2),('INT8',1)],
                                 value=2, description='Precision:'),
)

---
## 5 · Sliding Window Attention

Instead of every query attending to **all** past tokens, each query attends only to the **W nearest** tokens.  
This caps the KV cache size at $W$ entries per layer regardless of total sequence length.

Used by **Mistral 7B** and **GPT-OSS** variants.

In [ ]:
def draw_sliding_window(seq_len=16, window=5, current_query=10):
    current_query = min(current_query, seq_len - 1)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, title, use_window in [
        (axes[0], 'Full (causal) attention', False),
        (axes[1], f'Sliding window  W={window}', True),
    ]:
        mat = np.zeros((seq_len, seq_len))
        for q in range(seq_len):
            lo = max(0, q - window + 1) if use_window else 0
            mat[q, lo:q+1] = 0.5
        # highlight current query row
        q = current_query
        lo = max(0, q - window + 1) if use_window else 0
        mat[q, lo:q+1] = 1.0

        im = ax.imshow(mat, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Key position'); ax.set_ylabel('Query position')
        ax.axhline(current_query - 0.5, color='#1565C0', lw=1.5, ls='--')
        ax.axhline(current_query + 0.5, color='#1565C0', lw=1.5, ls='--')

        n_attended = current_query + 1 if not use_window else min(window, current_query + 1)
        ax.set_xlabel(f'Keys attended at query {current_query}: {n_attended}')

    plt.suptitle(f'Seq len={seq_len}, query={current_query}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

widgets.interact(
    draw_sliding_window,
    seq_len      = widgets.IntSlider(value=16, min=8,  max=32, step=1,  description='Seq len:'),
    window       = widgets.IntSlider(value=5,  min=2,  max=16, step=1,  description='Window W:'),
    current_query= widgets.IntSlider(value=10, min=0,  max=31, step=1,  description='Query idx:'),
)

### 5b · Streaming LLM

**Streaming LLM** keeps a small set of "sink" tokens at the very beginning (the model heavily attends to these even when they appear irrelevant) plus a sliding window of recent tokens.  
This avoids the attention-score collapse that appears when the window simply drops early tokens.

In [ ]:
def draw_streaming_llm(seq_len=20, n_sink=4, window=6, current_query=14):
    current_query = min(current_query, seq_len - 1)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, title, use_sink in [
        (axes[0], f'Sliding window only  W={window}', False),
        (axes[1], f'Streaming LLM  sink={n_sink}  W={window}', True),
    ]:
        mat = np.zeros((seq_len, seq_len))
        for q in range(seq_len):
            lo = max(0, q - window + 1)
            mat[q, lo:q+1] = 0.5
            if use_sink:
                mat[q, :min(n_sink, q+1)] = 0.75
        q = current_query
        lo = max(0, q - window + 1)
        mat[q, lo:q+1] = 1.0
        if use_sink:
            mat[q, :min(n_sink, q+1)] = 0.9

        im = ax.imshow(mat, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Key position'); ax.set_ylabel('Query position')
        ax.axhline(current_query - 0.5, color='#1565C0', lw=1.5, ls='--')
        ax.axhline(current_query + 0.5, color='#1565C0', lw=1.5, ls='--')
        if use_sink and n_sink > 0:
            ax.axvline(n_sink - 0.5, color='green', lw=1.5, ls=':')
            ax.text(n_sink/2 - 0.5, -0.9, 'sinks', ha='center', color='green', fontsize=9)

    plt.suptitle(f'Seq len={seq_len}, query={current_query}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

widgets.interact(
    draw_streaming_llm,
    seq_len      = widgets.IntSlider(value=20, min=10, max=40, step=1,  description='Seq len:'),
    n_sink       = widgets.IntSlider(value=4,  min=0,  max=10, step=1,  description='Sink tokens:'),
    window       = widgets.IntSlider(value=6,  min=2,  max=15, step=1,  description='Window W:'),
    current_query= widgets.IntSlider(value=14, min=0,  max=39, step=1,  description='Query idx:'),
)

---
## 6 · Pruning the KV Cache (Scissorhands / H2O)

Key observation: **most tokens receive very little attention**; only a small fraction ("heavy hitters") accumulate large attention scores repeatedly.

**Scissorhands** and **H2O** evict low-importance tokens from the KV cache to keep only the heavy hitters plus recent tokens.

The simulation below generates a random attention-score history and shows which tokens survive pruning.

In [ ]:
rng2 = np.random.default_rng(7)

def simulate_pruning(seq_len=20, keep_ratio=0.4, recent_tokens=3, seed=7):
    rng_local = np.random.default_rng(seed)
    # Simulate accumulated attention scores (heavy-tailed)
    # a few tokens are "heavy hitters"
    raw = rng_local.exponential(scale=1.0, size=seq_len)
    raw[:recent_tokens] = 0  # exclude recent from ranking (they're kept separately)
    acc_scores = raw / raw.sum()

    n_keep = max(1, int((seq_len - recent_tokens) * keep_ratio))
    ranked = np.argsort(acc_scores[:-recent_tokens])[::-1]
    heavy_hitters = set(ranked[:n_keep].tolist())
    recents       = set(range(seq_len - recent_tokens, seq_len))
    kept          = heavy_hitters | recents
    evicted       = set(range(seq_len)) - kept

    fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))

    # Accumulated attention scores
    colors = ['#2196F3' if i in recents else
              '#4CAF50' if i in heavy_hitters else
              '#EF5350'
              for i in range(seq_len)]
    axes[0].bar(range(seq_len), acc_scores, color=colors, edgecolor='white')
    axes[0].set_title('Accumulated attention scores per token')
    axes[0].set_xlabel('Token position')
    axes[0].set_ylabel('Score (normalised)')
    patches = [
        mpatches.Patch(color='#4CAF50', label='Heavy hitter (kept)'),
        mpatches.Patch(color='#2196F3', label=f'Recent (last {recent_tokens}, kept)'),
        mpatches.Patch(color='#EF5350', label='Evicted'),
    ]
    axes[0].legend(handles=patches, fontsize=8)

    # Cache state
    status = ['kept' if i in kept else 'evicted' for i in range(seq_len)]
    c2 = ['#4CAF50' if s == 'kept' else '#EF5350' for s in status]
    axes[1].bar(range(seq_len), [1]*seq_len, color=c2, edgecolor='white')
    axes[1].set_title(f'KV Cache after pruning — kept {len(kept)}/{seq_len} '
                      f'({100*len(kept)/seq_len:.0f}%)')
    axes[1].set_xlabel('Token position')
    axes[1].set_yticks([])

    plt.tight_layout()
    plt.show()

widgets.interact(
    simulate_pruning,
    seq_len      = widgets.IntSlider(value=20, min=10, max=60,  step=1,   description='Seq len:'),
    keep_ratio   = widgets.FloatSlider(value=0.4, min=0.1, max=1.0, step=0.05, description='Keep ratio:'),
    recent_tokens= widgets.IntSlider(value=3,  min=1,  max=10,  step=1,   description='Recent kept:'),
    seed         = widgets.IntSlider(value=7,  min=0,  max=99,  step=1,   description='Random seed:'),
)

---
## 7 · Cross-Conversation Cache Reuse

When the **system prompt** (or a long document prepended to every call) is identical across requests, a server can cache its KV representations and reuse them — charging only for the variable suffix.

**Key design rule:** put stable content first, variable content last — cache hits require a common prefix.

```
[System prompt — stable]  [Memory / tools — semi-stable]  [User message — variable]
  ←────────────── CACHE HIT ──────────────→  ← compute only this →
```

The widget below estimates the token savings for a batch of requests.

In [ ]:
def draw_prompt_cache(system_tokens=2000, user_tokens=200, n_requests=50, cost_per_1k=0.003):
    total_tokens_no_cache  = n_requests * (system_tokens + user_tokens)
    total_tokens_with_cache = system_tokens + n_requests * user_tokens  # system computed once
    saved_tokens = total_tokens_no_cache - total_tokens_with_cache
    saved_dollars = saved_tokens / 1000 * cost_per_1k

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Stacked bar per approach
    cats = ['No cache', 'Prefix cache']
    sys_vals  = [n_requests * system_tokens, system_tokens]
    user_vals = [n_requests * user_tokens,   n_requests * user_tokens]

    x = np.arange(2)
    axes[0].bar(x, sys_vals,  label='System tokens (computed)', color='#F44336', edgecolor='white')
    axes[0].bar(x, user_vals, bottom=sys_vals, label='User tokens', color='#42A5F5', edgecolor='white')
    axes[0].set_xticks(x); axes[0].set_xticklabels(cats)
    axes[0].set_ylabel('Total tokens processed')
    axes[0].set_title(f'{n_requests} requests')
    axes[0].legend(fontsize=9)
    for xi, (s, u) in enumerate(zip(sys_vals, user_vals)):
        axes[0].text(xi, (s+u)*1.01, f'{(s+u):,}', ha='center', fontsize=9)

    # Summary text
    axes[1].axis('off')
    summary = (
        f"Without cache : {total_tokens_no_cache:>12,} tokens\n"
        f"With prefix cache : {total_tokens_with_cache:>10,} tokens\n"
        f"─────────────────────────────────\n"
        f"Tokens saved  : {saved_tokens:>12,}\n"
        f"Cost saved    : ${saved_dollars:>11.2f}\n"
        f"  (@ ${cost_per_1k}/1k tokens)\n\n"
        f"Cache hit rate: {100*(1 - total_tokens_with_cache/total_tokens_no_cache):.1f}%"
    )
    axes[1].text(0.05, 0.5, summary, transform=axes[1].transAxes,
                 fontsize=12, va='center', fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='#E3F2FD', alpha=0.8))

    plt.suptitle('Prompt prefix caching — cost savings', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

widgets.interact(
    draw_prompt_cache,
    system_tokens = widgets.IntSlider(value=2000, min=100,  max=50000, step=100, description='System tokens:'),
    user_tokens   = widgets.IntSlider(value=200,  min=10,   max=5000,  step=10,  description='User tokens:'),
    n_requests    = widgets.IntSlider(value=50,   min=2,    max=1000,  step=1,   description='# Requests:'),
    cost_per_1k   = widgets.FloatSlider(value=0.003, min=0.0001, max=0.05, step=0.0005,
                                         readout_format='.4f', description='$/1k tokens:'),
)

---
## 8 · Summary Table

In [ ]:
html = """
<style>
  table{border-collapse:collapse;width:100%;font-size:13px}
  th{background:#1565C0;color:white;padding:8px 10px;text-align:left}
  tr:nth-child(even){background:#E3F2FD}
  td{padding:7px 10px;border-bottom:1px solid #ccc}
  .y{color:#2E7D32;font-weight:bold} .n{color:#B71C1C;font-weight:bold}
</style>
<table>
<tr><th>Method</th><th>Core idea</th><th>Changes Attention?</th><th>Needs fine-tuning?</th><th>Trade-off</th></tr>
<tr><td>Flash Attention</td><td>Move less data between HBM↔SRAM</td><td class='n'>No</td><td class='n'>No</td><td>Tiny extra compute + engineering</td></tr>
<tr><td><b>KV Cache</b></td><td>Store K,V; reuse across steps</td><td class='n'>No</td><td class='n'>No</td><td>Memory proportional to seq len</td></tr>
<tr><td>Multi-Query Attention</td><td>All queries share one K,V</td><td class='y'>Yes</td><td class='y'>Yes</td><td>May hurt model quality</td></tr>
<tr><td>Group-Query Attention</td><td>Groups share K,V (Llama, Gemma)</td><td class='y'>Yes</td><td class='y'>Yes</td><td>Good quality–memory trade-off</td></tr>
<tr><td>Multi-head Latent Attention</td><td>Compress K,V to low-rank latent (DeepSeek)</td><td class='y'>Yes</td><td class='y'>Yes</td><td>Tiny cache, strong performance</td></tr>
<tr><td>Sliding Window Attention</td><td>Attend only to W recent tokens</td><td class='y'>Yes</td><td>?</td><td>Loses long-range context</td></tr>
<tr><td>Streaming LLM</td><td>Sliding window + sink tokens</td><td class='y'>Yes</td><td>?</td><td>Better than pure sliding window</td></tr>
<tr><td>Pruning KV Cache</td><td>Evict low-attention tokens at runtime</td><td class='y'>Yes</td><td class='n'>No</td><td>May hurt quality for rare patterns</td></tr>
<tr><td>Speculative Decoding</td><td>Small model drafts, large model verifies</td><td class='n'>No (in theory)</td><td class='n'>No</td><td>Extra compute for small model</td></tr>
</table>
"""
display(HTML(html))

---
## 9 · Student Reflection Questions

1. **Memory vs. compute trade-off:** KV Cache saves compute but costs memory. At what sequence length does the KV cache become the bottleneck for a given GPU?

2. **GQA group size:** If you halve the group size in Group-Query Attention, what happens to (a) cache memory and (b) model quality? Why?

3. **Sliding window vs. full attention:** Name a task where sliding window attention would clearly fail. Name one where it would be fine.

4. **Sink tokens:** Why do language models attend heavily to the very first tokens even when they are irrelevant (e.g., the `<bos>` token)?

5. **Cross-conversation caching:** You're building an AI agent that uses a 10,000-token system prompt and receives short 50-token user messages. How should you structure the prompt to maximise cache hits?

6. **Pruning risk:** Scissorhands and H2O keep heavy-hitter tokens based on past attention. Can you think of a scenario where a "low-attention" token is actually critical for a later answer?

---
## 10 · Suggested Extensions

- **Implement a tiny GPT with KV Cache** using PyTorch and measure the wall-clock speedup vs. the naive version for sequences of length 64 → 1024.
- **Reproduce Figure 1 of the H2O paper:** plot perplexity vs. cache budget fraction for a small LM on a text dataset.
- **Benchmark MHA vs. GQA:** fine-tune a tiny transformer on a classification task with different group sizes and plot accuracy vs. inference memory.
- **Implement Streaming LLM:** feed a model tokens one-by-one and maintain the sink + sliding-window cache manually.
- **Explore prompt-caching APIs:** use the Anthropic or OpenAI API to measure actual latency reduction from prefix caching on a repeated system prompt.